# SERGIO simulations for TopGAE (Fig. 2C-D)

Generates the low-, medium- and high-density SERGIO datasets (5-10, 20-30 and 50-60 targets per TF; 60 TFs, 550 target genes, 5 cell types x 1,000 cells) and plots the dyngen / SERGIO results.

* Requires the SERGIO simulator (https://github.com/PayamDiba/SERGIO): clone it and add the cloned repository root to `PYTHONPATH` so that `from SERGIO.sergio import sergio` works.
* Set `TARGETS_PER_TF` in the GRN cell and run the notebook once per density.
* Outputs go to `sergio_data/` (`<GRN_NAME>_<TypeN>_mat.csv` and `<GRN_NAME>_net.csv`), which `run_sergio.py` reads.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy as sc

In [2]:
from SERGIO.sergio import sergio

In [ ]:
import random
from collections import defaultdict

def generate_fixed_k_grn(filename, num_tfs, num_genes, targets_per_tf, fixed_k, fixed_hill):
    """
    Generates a GRN where the ONLY variable is the network topology (who regulates whom).
    K and Hill are constants to ensure they do not influence correlation.
    """
    # target_id -> list of (regulator_id, k, hill)
    gene_map = defaultdict(list)
    
    tf_ids = list(range(num_tfs))
    target_gene_ids = list(range(num_tfs, num_tfs + num_genes))
    
    # Track which genes have been assigned at least once to ensure no "dead" genes
    assigned_genes = set()

    for tf_id in tf_ids:
        # 1. Determine fan-out (number of genes this TF controls)
        n_targets = random.randint(targets_per_tf[0], targets_per_tf[1])
        
        # 2. Randomly select the targets for this TF
        selected_targets = random.sample(target_gene_ids, n_targets)
        
        # 3. Apply the FIXED parameters to every gene in this TF's regulon
        for target_id in selected_targets:
            gene_map[target_id].append((tf_id, fixed_k, fixed_hill))
            assigned_genes.add(target_id)

    # 4. Cleanup: If any genes were missed by random sampling, assign them to a random TF
    # This prevents the simulator from crashing or having "zero-expression" features
    unassigned = set(target_gene_ids) - assigned_genes
    for target_id in unassigned:
        random_tf = random.choice(tf_ids)
        gene_map[target_id].append((random_tf, fixed_k, fixed_hill))

    # 5. Format for CSV: [target_id, n_regs, TFs..., Ks..., Hills...]
    rows = []
    for target_id in sorted(gene_map.keys()):
        regs_data = gene_map[target_id]
        n_regs = len(regs_data)
        
        tfs = [float(r[0]) for r in regs_data]
        ks = [float(r[1]) for r in regs_data]
        hills = [float(r[2]) for r in regs_data]
        
        row = [float(target_id), float(n_regs)] + tfs + ks + hills
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False, header=False)
    print(f"Success: {filename} generated.")
    print(f"Stats: {num_tfs} TFs, {num_genes} Genes. Constant K={fixed_k}, Hill={fixed_hill}")

# --- EXPERIMENTAL RUNS ---
# Network density (targets per TF). The paper uses three settings; run the
# notebook once per setting:  low (5, 10), medium (20, 30), high (50, 60).
TARGETS_PER_TF = (50, 60)
GRN_NAME = f"sergio_{TARGETS_PER_TF[0]}_{TARGETS_PER_TF[1]}_target"
DATA_DIR = "sergio_data"  # expression + network CSVs read by run_sergio.py
os.makedirs(DATA_DIR, exist_ok=True)

generate_fixed_k_grn(
    filename=f"{GRN_NAME}.csv",
    num_tfs=60,
    num_genes=550,
    targets_per_tf=TARGETS_PER_TF,
    fixed_k=2.5, 
    fixed_hill=2.0
)

In [15]:
def generate_basal(num_tfs, num_cell_types):

    tf_ids = list(range(num_tfs))
    random.shuffle(tf_ids)

    # Biological Proportions
    hk_count   = int(num_tfs * 0.20)
    rand_count = int(num_tfs * 0.30)
    spec_count = num_tfs - hk_count - rand_count

    hk_ids   = tf_ids[:hk_count]
    rand_ids = tf_ids[hk_count:hk_count + rand_count]
    spec_ids = tf_ids[hk_count + rand_count:]

    all_rows = []

    # 1. Housekeeping: Stable baseline
    # No change between modes - HK should remain a control variable.
    for tf_id in hk_ids:
        base_rate = np.random.lognormal(mean=0.8, sigma=0.1) * 10.0
        row = [float(tf_id)] + [round(base_rate, 3)] * num_cell_types
        all_rows.append(row)

    # 2. Random/Noisy: Background interference
    for tf_id in rand_ids:
        rates = []
        for _ in range(num_cell_types):
            # In LOW variance mode, we increase noise to further blur the classes
            dropout_prob = 0.6
            scale_val = 0.5
            
            if random.random() < dropout_prob:
                rates.append(0.0)
            else:
                rates.append(round(np.random.exponential(scale=scale_val) * 10.0, 3))
        row = [float(tf_id)] + rates
        all_rows.append(row)

    # 3. Cell-Type Specific (CTS): The Driver of Variance
    index = 0
    tfs_per_type = spec_count // num_cell_types
    
    for ct in range(num_cell_types):
        count = tfs_per_type + (1 if ct < (spec_count % num_cell_types) else 0)
        batch = spec_ids[index:index + count]
        index += count

        for tf_id in batch:
            # HIGH VARIANCE: Binary-like behavior (Master Regulators)
            # Primary is very high, others are practically zero
            rates = [round(random.uniform(0.0, 0.01), 3) for _ in range(num_cell_types)]
            rates[ct] = round(np.random.gamma(shape=5.0, scale=4.0) * 10.0, 3) # High Mean
            
            # Sharp lineage transition (minimal leaking)
            neighbor = (ct + 1) % num_cell_types
            rates[neighbor] = round(rates[ct] * random.uniform(0.05, 0.1), 3)
            row = [float(tf_id)] + rates
            all_rows.append(row)

    all_rows.sort(key=lambda x: x[0])
    df = pd.DataFrame(all_rows)
    filename = f"sergio_basal_regs.csv"
    df.to_csv(filename, index=False, header=False)
    print(f"Generated {filename} with {num_tfs} TFs.")

# Generate both sets for comparison
generate_basal(num_tfs=60, num_cell_types=5)

Generated sergio_basal_regs.csv with 60 TFs.


In [ ]:
sim = sergio(number_genes=60+550, number_bins = 5, number_sc = 1000, noise_params = 1, decays=0.8, sampling_state=15, noise_type='dpd')
sim.build_graph(input_file_taregts=f'{GRN_NAME}.csv', input_file_regs='sergio_basal_regs.csv', shared_coop_state=2)
sim.simulate()
expr = sim.getExpressions()
expr_clean = np.concatenate(expr, axis = 1)

In [36]:
"""
Add outlier genes
"""
expr_O = sim.outlier_effect(expr, outlier_prob = 0.01, mean = 0.8, scale = 1)

"""
Add Library Size Effect
"""
libFactor, expr_O_L = sim.lib_size_effect(expr_O, mean = 4.6, scale = 0.4)

"""
Add Dropouts
"""
binary_ind = sim.dropout_indicator(expr_O_L, shape = 6.5, percentile = 82)
expr_O_L_D = np.multiply(binary_ind, expr_O_L)

"""
Convert to UMI count
"""
count_matrix = sim.convert_to_UMIcounts(expr_O_L_D)

"""
Make a 2d gene expression matrix
"""
count_matrix = np.concatenate(count_matrix, axis = 1)
count_matrix

array([[0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 1],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(610, 5000))

In [37]:
import anndata as ad
import scanpy as sc
import pandas as pd

gene_names = [f"Gene_{i+1}" for i in range(610)]
cell_names = [f"Cell_{i+1}" for i in range(5000)]
cell_types = [f"Type{(i // 1000) + 1}" for i in range(5000)]

adata = ad.AnnData(
    X=count_matrix.T,
    obs=pd.DataFrame(
        {
            "cell_type": cell_types
        },
        index=cell_names
    ),
    var=pd.DataFrame(index=gene_names)
)
adata.layers["counts"] = adata.X.copy()
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)
adata

AnnData object with n_obs × n_vars = 5000 × 610
    obs: 'cell_type'
    uns: 'log1p'
    layers: 'counts'

In [ ]:
counts = adata.layers["counts"]
counts = np.asarray(counts)
counts_df = pd.DataFrame(counts, index=adata.obs_names, columns=adata.var_names)
counts_df.index.name = "cell"

for ct, cells in adata.obs.groupby("cell_type", observed=True).groups.items():
    sub = counts_df.loc[cells]
    path = os.path.join(DATA_DIR, f"{GRN_NAME}_{ct}_mat.csv")
    sub.to_csv(path)
    print(f"{ct}: {sub.shape[0]} cells x {sub.shape[1]} genes -> {path}")

In [39]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=['cell_type'],
    legend_loc="on data",
    show=True,
    save=f"{GRN_NAME}_umap.pdf"
)

In [ ]:
import csv
import pandas as pd

def sergio_grn_to_edges(filename, gene_name=lambda i: f"Gene_{i+1}"):
    """Convert a SERGIO interaction file to a from/to edge table.

    Each row is: target_id, n_regs, TF_1..TF_n, K_1..K_n, Hill_1..Hill_n
    """
    edges = []
    with open(filename) as f:
        for line in csv.reader(f):
            vals = [float(v) for v in line if v.strip() != ""]   # drop padding from ragged rows
            if not vals:
                continue
            target, n = int(vals[0]), int(vals[1])
            tfs   = vals[2:2 + n]
            ks    = vals[2 + n:2 + 2 * n]
            hills = vals[2 + 2 * n:2 + 3 * n]
            for tf, k, h in zip(tfs, ks, hills):
                edges.append({"from": gene_name(int(tf)), "to": gene_name(target),
                              "K": k, "Hill": h})
    return pd.DataFrame(edges)


net = sergio_grn_to_edges(f"{GRN_NAME}.csv")
print(net.head())
print(f"{len(net)} edges, {net['from'].nunique()} regulators, {net['to'].nunique()} targets")

net.to_csv(os.path.join(DATA_DIR, f"{GRN_NAME}_net.csv"), index=False)

## Figures (Fig. 2B and 2D)

Run `run_dyngen.py` and `run_sergio.py` from this folder first; the cells below read their per-run result tables.

In [11]:
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
dyngen_df = pd.read_csv('Dyngen_TopGAE_Results/Per_Run.csv')
sergio_df = pd.read_csv('SERGIO_TopGAE_Results/Per_Run.csv')
dyngen_df

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(5.5, 3.5))

sns.violinplot(
    data=dyngen_df,
    x="Dataset",
    y="AUROC",
    inner="box",
    palette="colorblind",
    gap=0.2,
    width=0.8,
    ax=ax,
)


ax.set_xlabel("Dataset")
ax.set_ylabel("AUROC")
ax.set_title("Dyngen Results")
# ax.legend(title="Seed", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig('dyngen_auroc_plot.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.show()

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=sergio_df,
    x="GRN",
    hue="CellType",
    y="AUPRC",
    inner="box",
    palette="colorblind",
    gap=0.2,
    width=0.8,
    ax=ax,
    order=["5_10_target", "20_30_target", "50_60_target"]
)


ax.set_xlabel("GRN")
ax.set_ylabel("AUPRC")
ax.set_title("Sergio Results")
# ax.legend(title="Seed", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig('sergio_auprc_plot.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.show()